<div class="alert alert-block alert-info">
Before you start running this notebook, create a cloud app in Verily Workbench with a <b>low compute</b> profile. To do so,
<br/>
    
<ul>
    <li>Open the <b>Apps</b> tab in your workspace and click <b>+ New app instance</b>.</li>
    <li>Select <b>R Analysis</b>.</li>
    <li>On the <b>Compute options</b> step, the default profile (2 CPUs, 13 GB of memory) is sufficient.</li>
    <li>For more information, see the <a href="https://support.workbench.verily.com/docs/guides/cloud_apps/gcp/compute_config_options/">Verily Workbench compute profile guide</a>.</li>
</ul>
    
</div>

<h1>Notebook Objectives</h1>

This notebook reviews haplotype-resolved assembly QC for Phase 1 samples. Using the Terra assembly-analysis table, it scatter-plots hap1/hap2 contig count, N50, duplication ratio, and total length against mosdepth coverage.

<h2>About notebook public versions</h2>

This notebook is a public counterpart of an analysis notebook in the *All of Us* Researcher Workbench (Controlled Tier). Individual-level genomic, EHR, and derived files are not included in this repository. In these public notebook versions, output cells have also been removed.

For run details, see the version of this notebook in the Researcher Workbench Controlled Tier workspace.

In [ ]:
library(reticulate)
library(bigrquery)
library(ggplot2)
library(tidyverse)
library(ggpubr)
library(vcfR)

In [ ]:
# The FISS (FireCloud) Python API, called from R through reticulate, for reading Terra data tables.
fapi = import('firecloud.api')

In [ ]:
# Workspace details Terra sets as environment variables.
BILLING_PROJECT_ID = Sys.getenv('GOOGLE_PROJECT')       # the Google billing project
WORKSPACE_BUCKET = Sys.getenv('WORKSPACE_BUCKET')       # the workspace bucket
WORKSPACE_NAME = Sys.getenv('WORKSPACE_NAME')           # workspace name
WORKSPACE_NAMESPACE = Sys.getenv('WORKSPACE_NAMESPACE') # workspace namespace

In [ ]:
# Load a Terra data table into a tibble, converting 'true'/'false' list columns to logicals.
load_workspace_table <- function(table_name) {
    a = fapi$get_entities(WORKSPACE_NAMESPACE, WORKSPACE_NAME, table_name)

    json = a$json()
    df = tibble(metadata = json) %>% unnest_wider(metadata) %>% unnest_wider(attributes)

    for (name in df %>% select(where(is.list)) %>% names()) {
        q = as.vector(unlist(df[,name])) %in% c("true", "TRUE", "false", "FALSE") %>% unique()
        if (q) {
            df[,name] = as.logical(as.vector(unlist(df[,name])))
        }
    }
    
    return(df)
}

In [ ]:
# Per-sample assembly statistics (contig count, N50, duplication ratio, total length per haplotype) and
# read coverage (mosdepth_cov).
ha_sample_asm_analysis_df = load_workspace_table('ha-sample-asm-analysis')

In [ ]:
ha_sample_asm_analysis_df

In [ ]:
# Number of contigs vs coverage, per haplotype.
par(mar=c(5, 8, 4, 2))

plot(ha_sample_asm_analysis_df$mosdepth_cov, ha_sample_asm_analysis_df$hap1_num_contigs,
    bty="n",
    las=1,
    xlab="Mean Coverage",
    ylab="",
    cex=1.3,
    cex.axis=1.3,
    cex.lab=1.3,
    pch=21,
    col="#ff0000",
)

points(ha_sample_asm_analysis_df$mosdepth_cov, ha_sample_asm_analysis_df$hap2_num_contigs,
    pch=21,
    col="#0000ff"
)

mtext("Number of contigs", 2, line = 7, cex=1.3)

legend("topright", c("Haplotype 1", "Haplotype 2"), pch=21, col=c("#ff0000", "#0000ff"))

In [ ]:
# Contig N50 vs coverage, per haplotype.
par(mar=c(5, 8, 4, 2))

plot(ha_sample_asm_analysis_df$mosdepth_cov, ha_sample_asm_analysis_df$hap1_n50,
    bty="n",
    las=1,
    xlab="Mean Coverage",
    ylab="",
    cex=1.3,
    cex.axis=1.3,
    cex.lab=1.3,
    pch=21,
    col="#ff0000",
)

points(ha_sample_asm_analysis_df$mosdepth_cov, ha_sample_asm_analysis_df$hap2_n50,
    pch=21,
    col="#0000ff"
)

mtext("Contig N50", 2, line = 7, cex=1.3)

legend("topleft", c("Haplotype 1", "Haplotype 2"), pch=21, col=c("#ff0000", "#0000ff"))

In [ ]:
# Mean haplotype-2 N50.
ha_sample_asm_analysis_df$hap2_n50 %>% na.omit() %>% mean()

In [ ]:
# Duplication ratio vs coverage, per haplotype.
par(mar=c(5, 8, 4, 2))

plot(ha_sample_asm_analysis_df$mosdepth_cov, ha_sample_asm_analysis_df$hap1_duplication_ratio,
    bty="n",
    las=1,
    xlab="Mean Coverage",
    ylab="",
    cex=1.3,
    cex.axis=1.3,
    cex.lab=1.3,
    pch=21,
    col="#ff0000",
)

points(ha_sample_asm_analysis_df$mosdepth_cov, ha_sample_asm_analysis_df$hap2_duplication_ratio,
    pch=21,
    col="#0000ff"
)

mtext("Duplication Ratio", 2, line = 7, cex=1.3)

legend("topright", c("Haplotype 1", "Haplotype 2"), pch=21, col=c("#ff0000", "#0000ff"))

In [ ]:
# Total assembled length vs coverage, per haplotype.
par(mar=c(5, 8, 4, 2))

plot(ha_sample_asm_analysis_df$mosdepth_cov, ha_sample_asm_analysis_df$hap1_total_length,
    bty="n",
    las=1,
    xlab="Mean Coverage",
    ylab="",
    cex=1.3,
    cex.axis=1.3,
    cex.lab=1.3,
    pch=21,
    col="#ff0000",
)

points(ha_sample_asm_analysis_df$mosdepth_cov, ha_sample_asm_analysis_df$hap2_total_length,
    pch=21,
    col="#0000ff"
)

mtext("Total haplotype length", 2, line = 7, cex=1.3)

legend("topright", c("Haplotype 1", "Haplotype 2"), pch=21, col=c("#ff0000", "#0000ff"))